In [1]:
from google.colab import drive

drive.mount('/content/drive')




Mounted at /content/drive


In [2]:
from pathlib import Path

ruta_proyecto = Path(
    "/content/drive/MyDrive/AI Engineering/ai-entity-extraction-pipeline"
)

print("¿Existe la carpeta?", ruta_proyecto.exists())

¿Existe la carpeta? True


In [3]:
%cd /content/drive/MyDrive/AI\ Engineering/ai-entity-extraction-pipeline

/content/drive/MyDrive/AI Engineering/ai-entity-extraction-pipeline


In [ ]:
## Definimos schemas.py

%%writefile schemas.py

from enum import Enum

from pydantic import BaseModel, Field


class NivelCriticidad(str, Enum):
    BAJA = "baja"
    MEDIA = "media"
    ALTA = "alta"


class ExtraccionTecnica(BaseModel):
    tecnologias: list[str] = Field(
        min_length=1,
        description="Lista de tecnologías identificadas en el texto."
    )

    nivel_de_criticidad: NivelCriticidad = Field(
        description="Nivel de criticidad: baja, media o alta."
    )

    resumen_tecnico: str = Field(
        min_length=1,
        description="Resumen breve del problema técnico."
    )



Overwriting schemas.py


In [ ]:
from schemas import ExtraccionTecnica, NivelCriticidad

resultado = ExtraccionTecnica(
    tecnologias=["FastAPI", "Redis", "PostgreSQL"],
    nivel_de_criticidad=NivelCriticidad.ALTA,
    resumen_tecnico="Problemas de conexiones concurrentes."
)

print(resultado.model_dump_json(indent=2))

{
  "tecnologias": [
    "FastAPI",
    "Redis",
    "PostgreSQL"
  ],
  "nivel_de_criticidad": "alta",
  "resumen_tecnico": "Problemas de conexiones concurrentes."
}


In [ ]:
## Vamos a generar un error con los valores permitidos por Pydantic

try:
    resultado_invalido = ExtraccionTecnica(
        tecnologias=["FastAPI"],
        nivel_de_criticidad="urgente",
        resumen_tecnico="Problemas de conexión."
    )
except Exception as e:
    print(type(e).__name__)
    print(e)

ValidationError
1 validation error for ExtraccionTecnica
nivel_de_criticidad
  Input should be 'baja', 'media' or 'alta' [type=enum, input_value='urgente', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/enum


In [ ]:
## Ahora pruebo con una lista vacía
try:
    resultado_invalido = ExtraccionTecnica(
        tecnologias=[],
        nivel_de_criticidad=NivelCriticidad.MEDIA,
        resumen_tecnico="Problemas de conexión."
    )
except Exception as e:
    print(type(e).__name__)
    print(e)

ValidationError
1 validation error for ExtraccionTecnica
tecnologias
  List should have at least 1 item after validation, not 0 [type=too_short, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.14/v/too_short


In [4]:
%pip install -U langchain-openai

import pydantic
import langchain
import langchain_openai

print("Pydantic:", pydantic.__version__)
print("LangChain:", langchain.__version__)
print("langchain-openai: instalado correctamente")

Pydantic: 2.14.1
LangChain: 1.4.7
langchain-openai: instalado correctamente


In [5]:
## Comprobamos si tenemos acceso a la key de OpenAI

from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")

print("API key configurada:", bool(api_key))

API key configurada: True


In [6]:
## Traemos la variable de entorno con la key

import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [7]:
## Definción chain.py

%%writefile chain.py

import logging
import os

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

from schemas import ExtraccionTecnica


# Configuración de logs
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


# Configurar el modelo
model = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0
)


# Definir las instrucciones para el modelo
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Eres un analista especializado en infraestructura y software.

        Analiza el texto técnico recibido y extrae la siguiente información:

        1. tecnologias:
           Identifica las tecnologías mencionadas explícitamente.
           No inventes tecnologías que no estén respaldadas por el texto.

        2. nivel_de_criticidad:
           Clasifica la criticidad como baja, media o alta.
           Considera el impacto, la gravedad y la urgencia descritos.
           No supongas que existe un impacto grave si el texto no lo indica.

        3. resumen_tecnico:
           Resume brevemente el problema técnico principal.

        Utiliza únicamente la información disponible en el texto.
        Si la información es ambigua, evita inventar detalles.
        """
    ),
    (
        "human",
        "Analiza el siguiente texto técnico:\n\n{text}"
    )
])

# Conectar el modelo con el esquema Pydantic
structured_model = model.with_structured_output(
    ExtraccionTecnica,
    include_raw=True
)

# Función de validación de respuesta
def validar_respuesta(resultado):

    if resultado["parsing_error"] is not None:
        raise resultado["parsing_error"]

    if resultado["parsed"] is None:
        raise ValueError("El modelo no devolvió una respuesta válida.")

    return resultado["parsed"]

# Construir la cadena LCEL
chain = (
    prompt
    | structured_model
    | RunnableLambda(validar_respuesta)
).with_retry(
    retry_if_exception_type=(Exception,),
    stop_after_attempt=3
)

# Función pública para procesar textos técnicos

async def process_text(text: str) -> ExtraccionTecnica:
    """Procesa un texto técnico y devuelve una extracción validada."""

    logger.info("Iniciando extracción técnica")

    try:
        resultado = await chain.ainvoke({"text": text})

        logger.info("Extracción técnica completada correctamente")
        return resultado

    except Exception:
        logger.exception("Falló el procesamiento del texto")
        raise


Overwriting chain.py


In [ ]:
## Verificación de que el prompt funcional

from chain import prompt

print(prompt.input_variables)

['text']


In [ ]:
from chain import chain

print("Tipo de cadena:", type(chain).__name__)

Tipo de cadena: RunnableSequence


In [ ]:
import importlib
import chain as chain_module

importlib.reload(chain_module)

print("Cadena definida:", hasattr(chain_module, "chain"))
print("Tipo de cadena:", type(chain_module.chain).__name__)

Cadena definida: True
Tipo de cadena: RunnableSequence


In [ ]:
texto = """
La aplicación FastAPI presenta errores de conexión con PostgreSQL.
El pool de conexiones se agota durante los períodos de alta demanda,
provocando interrupciones en el servicio.
"""

resultado = await chain.ainvoke({"text": texto})

print(resultado)

tecnologias=['FastAPI', 'PostgreSQL'] nivel_de_criticidad=<NivelCriticidad.MEDIA: 'media'> resumen_tecnico='La aplicación FastAPI tiene problemas de conexión con PostgreSQL debido al agotamiento del pool de conexiones en momentos de alta demanda, lo que causa interrupciones en el servicio.'


In [ ]:
import importlib
import chain as chain_module

importlib.reload(chain_module)

texto = """
La aplicación FastAPI presenta errores de conexión con PostgreSQL.
El pool de conexiones se agota durante los períodos de alta demanda,
provocando interrupciones en el servicio.
"""

resultado = await chain_module.process_text(texto)

print(resultado.model_dump_json(indent=2))

{
  "tecnologias": [
    "FastAPI",
    "PostgreSQL"
  ],
  "nivel_de_criticidad": "alta",
  "resumen_tecnico": "La aplicación FastAPI tiene problemas de conexión con PostgreSQL debido al agotamiento del pool de conexiones en momentos de alta demanda, lo que causa interrupciones en el servicio."
}


In [ ]:

from langchain_core.runnables import RunnableLambda

intentos = {"cantidad": 0}

def operacion_de_prueba(entrada):
    intentos["cantidad"] += 1

    print(f"Ejecutando intento {intentos['cantidad']}")

    if intentos["cantidad"] < 3:
        raise ValueError("Error simulado")

    return "Operación completada correctamente"


prueba = RunnableLambda(operacion_de_prueba).with_retry(
    retry_if_exception_type=(ValueError,),
    stop_after_attempt=3,
    wait_exponential_jitter=False
)

resultado_prueba = prueba.invoke("iniciar")

print(resultado_prueba)
print("Total de intentos:", intentos["cantidad"])


Ejecutando intento 1
Ejecutando intento 2
Ejecutando intento 3
Operación completada correctamente
Total de intentos: 3


In [ ]:
from pydantic import ValidationError

print("Clase de excepción de Pydantic:", ValidationError.__name__)

Clase de excepción de Pydantic: ValidationError


In [ ]:
from pydantic import ValidationError
from schemas import ExtraccionTecnica

try:
    ExtraccionTecnica(
        tecnologias=[],
        nivel_de_criticidad="urgente",
        resumen_tecnico=""
    )

except ValidationError as e:
    print("Se detectó un error de validación:")
    print(type(e).__name__)
    print(e)

Se detectó un error de validación:
ValidationError
3 validation errors for ExtraccionTecnica
tecnologias
  List should have at least 1 item after validation, not 0 [type=too_short, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.14/v/too_short
nivel_de_criticidad
  Input should be 'baja', 'media' or 'alta' [type=enum, input_value='urgente', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/enum
resumen_tecnico
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_short


In [ ]:
from pydantic import ValidationError
from langchain_core.runnables import RunnableLambda

# Contador para saber cuántas veces se ejecuta la operación
intentos = {"cantidad": 0}

def operacion_de_prueba(entrada):
    intentos["cantidad"] += 1

    print(f"Intento número {intentos['cantidad']}")

    if intentos["cantidad"] == 1:
        # Generamos deliberadamente un ValidationError
        from schemas import ExtraccionTecnica

        ExtraccionTecnica(
            tecnologias=[],
            nivel_de_criticidad="urgente",
            resumen_tecnico=""
        )

    return "Operación completada correctamente"


prueba = RunnableLambda(operacion_de_prueba).with_retry(
    retry_if_exception_type=(ValidationError,),
    stop_after_attempt=3,
    wait_exponential_jitter=False
)

resultado = prueba.invoke("iniciar")

print("Resultado:", resultado)
print("Total de intentos:", intentos["cantidad"])

Intento número 1
Intento número 2
Resultado: Operación completada correctamente
Total de intentos: 2


In [ ]:
## Vamos a probar con nuestro modelo usando las API

from chain import process_text

texto = """
La aplicación de gestión de pedidos utiliza FastAPI y PostgreSQL.
Durante los períodos de alta demanda, el pool de conexiones a la base
de datos se agota y provoca interrupciones del servicio.
El equipo de soporte está investigando el problema.
"""

resultado = await process_text(texto)

print(resultado)



tecnologias=['FastAPI', 'PostgreSQL'] nivel_de_criticidad=<NivelCriticidad.MEDIA: 'media'> resumen_tecnico='El pool de conexiones a la base de datos se agota durante períodos de alta demanda, causando interrupciones del servicio.'


In [ ]:
## Comprobamos el tipo de objecto que entrega el contrato

from schemas import ExtraccionTecnica

print("Tipo de resultado:", type(resultado).__name__)
print("¿Es una instancia de ExtraccionTecnica?:",
      isinstance(resultado, ExtraccionTecnica))

Tipo de resultado: ExtraccionTecnica
¿Es una instancia de ExtraccionTecnica?: True


In [ ]:
from pydantic import ValidationError
from langchain_core.runnables import RunnableLambda
from schemas import ExtraccionTecnica

intentos = {"cantidad": 0}

def simular_respuesta(entrada):
    intentos["cantidad"] += 1

    print(f"Intento número {intentos['cantidad']}")

    if intentos["cantidad"] == 1:
        # Primera respuesta: datos inválidos
        return ExtraccionTecnica(
            tecnologias=[],
            nivel_de_criticidad="urgente",
            resumen_tecnico=""
        )

    # Segunda respuesta: datos válidos
    return ExtraccionTecnica(
        tecnologias=["FastAPI", "PostgreSQL"],
        nivel_de_criticidad="media",
        resumen_tecnico="Agotamiento del pool de conexiones."
    )

prueba = RunnableLambda(simular_respuesta).with_retry(
    retry_if_exception_type=(ValidationError,),
    stop_after_attempt=3,
    wait_exponential_jitter=False
)

resultado_prueba = prueba.invoke("texto técnico")

print("Resultado:", resultado_prueba)
print("Total de intentos:", intentos["cantidad"])

Intento número 1
Intento número 2
Resultado: tecnologias=['FastAPI', 'PostgreSQL'] nivel_de_criticidad=<NivelCriticidad.MEDIA: 'media'> resumen_tecnico='Agotamiento del pool de conexiones.'
Total de intentos: 2


In [ ]:
import langchain
import langchain_openai

print("LangChain:", langchain.__version__)
print("LangChain OpenAI:", langchain_openai.__version__)

LangChain: 1.4.7
LangChain OpenAI: 1.7.3


In [10]:
import importlib
import chain

importlib.reload(chain)

<module 'chain' from '/content/drive/MyDrive/AI Engineering/ai-entity-extraction-pipeline/chain.py'>

In [12]:
texto = """
La aplicación FastAPI presenta errores de conexión con PostgreSQL.
El pool de conexiones se agota durante los períodos de alta demanda,
provocando interrupciones en el servicio.
"""

resultado = await chain.process_text(texto)

print(resultado)
print("\nTipo de resultado:", type(resultado).__name__)

tecnologias=['FastAPI', 'PostgreSQL'] nivel_de_criticidad=<NivelCriticidad.ALTA: 'alta'> resumen_tecnico='La aplicación FastAPI tiene problemas de conexión con PostgreSQL debido al agotamiento del pool de conexiones en momentos de alta demanda, lo que causa interrupciones en el servicio.'

Tipo de resultado: ExtraccionTecnica


In [14]:
## Generamos un error en el primer intenteto y en el segundo tenemos un pyndantic correcto.

from langchain_core.runnables import RunnableLambda
from pydantic import ValidationError
from schemas import ExtraccionTecnica
from pydantic import ValidationError
from langchain_core.runnables import RunnableLambda

intentos = 0

def simular_procesamiento(texto):
    global intentos
    intentos += 1

    print(f"Intento número {intentos}")

    if intentos == 1:
        # Provocamos un error de validación intencional
        return ExtraccionTecnica(
            tecnologias=[],
            nivel_de_criticidad="urgente",
            resumen_tecnico=""
        )

    # En el segundo intento devolvemos una respuesta válida
    return ExtraccionTecnica(
        tecnologias=["FastAPI", "PostgreSQL"],
        nivel_de_criticidad="media",
        resumen_tecnico="Problemas de conexión con PostgreSQL."
    )


prueba_retry = RunnableLambda(simular_procesamiento).with_retry(
    retry_if_exception_type=(ValidationError,),
    stop_after_attempt=3
)

resultado_prueba = prueba_retry.invoke("Texto técnico de prueba")

print("\nResultado final:")
print(resultado_prueba)
print("\nTotal de intentos:", intentos)

Intento número 1
Intento número 2

Resultado final:
tecnologias=['FastAPI', 'PostgreSQL'] nivel_de_criticidad=<NivelCriticidad.MEDIA: 'media'> resumen_tecnico='Problemas de conexión con PostgreSQL.'

Total de intentos: 2


In [15]:
# Revisamos que tenemos en nuestra carpeta
import os

carpeta = "/content/drive/MyDrive/AI Engineering/ai-entity-extraction-pipeline"

for archivo in sorted(os.listdir(carpeta)):
    print(archivo)

__pycache__
ai-entity-extraction-pipeline.ipynb
chain.py
schemas.py


In [17]:
## Escribimos el archivo test_pipeline.py

%%writefile "/content/drive/MyDrive/AI Engineering/ai-entity-extraction-pipeline/test_pipeline.py"

import asyncio

from schemas import ExtraccionTecnica
from chain import process_text


async def test_process_text():

    texto = """
    La aplicación FastAPI presenta errores de conexión con PostgreSQL.
    El pool de conexiones se agota durante los períodos de alta demanda,
    provocando interrupciones en el servicio.
    """

    resultado = await process_text(texto)

    assert isinstance(resultado, ExtraccionTecnica)
    assert len(resultado.tecnologias) > 0
    assert resultado.nivel_de_criticidad.value in ["baja", "media", "alta"]
    assert len(resultado.resumen_tecnico) > 0

    print("Todas las verificaciones fueron superadas.")


if __name__ == "__main__":
    asyncio.run(test_process_text())

Writing /content/drive/MyDrive/AI Engineering/ai-entity-extraction-pipeline/test_pipeline.py


In [18]:
%cd "/content/drive/MyDrive/AI Engineering/ai-entity-extraction-pipeline"

!python test_pipeline.py

/content/drive/MyDrive/AI Engineering/ai-entity-extraction-pipeline
INFO:langchain_openai.chat_models._client_utils:langchain-openai detected system proxy configuration and no explicit `http_socket_options` / `http_client` / `http_async_client` / `openai_proxy`; skipping the custom `httpx` transport so httpx's env-proxy auto-detection applies. Pass `http_socket_options=[...]` to opt back into kernel-level TCP keepalive tuning on top of the env proxy.
INFO:chain:Iniciando extracción técnica
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:chain:Extracción técnica completada correctamente
Todas las verificaciones fueron superadas.


In [19]:
## Armando el requirements.txt
%%writefile "/content/drive/MyDrive/AI Engineering/ai-entity-extraction-pipeline/requirements.txt"

langchain
langchain-openai
pydantic

Writing /content/drive/MyDrive/AI Engineering/ai-entity-extraction-pipeline/requirements.txt


In [20]:
## Creamos el readme.txt


%%writefile "/content/drive/MyDrive/AI Engineering/ai-entity-extraction-pipeline/README.md"

# AI Entity Extraction Pipeline

## 1. Descripción del proyecto

Pipeline de procesamiento de textos técnicos desarrollado con Python, LangChain y Pydantic.

Su objetivo es analizar descripciones de arquitecturas de software, incidentes y errores técnicos para extraer información estructurada y validada.

## 2. Información extraída

El pipeline genera tres campos:

- **tecnologias:** lista de tecnologías identificadas en el texto.
- **nivel_de_criticidad:** clasificación en baja, media o alta.
- **resumen_tecnico:** descripción breve del problema técnico principal.

## 3. Arquitectura

El procesamiento sigue esta secuencia:

1. Recepción del texto técnico.
2. Construcción del prompt mediante ChatPromptTemplate.
3. Procesamiento con un modelo de OpenAI.
4. Generación de una respuesta estructurada según el esquema Pydantic.
5. Validación de los datos.
6. Reintentos automáticos ante excepciones.

La cadena se implementa mediante LCEL (LangChain Expression Language).

## 4. Estructura del proyecto

```text
ai-entity-extraction-pipeline/
│
├── ai-entity-extraction-pipeline.ipynb
├── schemas.py
├── chain.py
├── test_pipeline.py
├── requirements.txt
└── README.md
```

## 5. Requisitos e instalación

Se requiere Python 3.12 y una API Key de OpenAI.

Instalar las dependencias:

```bash
pip install -r requirements.txt
```

## 6. Configuración

Configurar la variable de entorno:

```bash
OPENAI_API_KEY=tu_api_key
```

No compartir ni publicar la API Key en el repositorio.

## 7. Uso

Ejemplo de utilización:

```python
import asyncio
from chain import process_text

texto = """
La aplicación FastAPI presenta errores de conexión con PostgreSQL.
El pool de conexiones se agota durante los períodos de alta demanda,
provocando interrupciones en el servicio.
"""

resultado = asyncio.run(process_text(texto))

print(resultado.model_dump_json(indent=2))
```

## 8. Ejemplo de salida

```json
{
  "tecnologias": [
    "FastAPI",
    "PostgreSQL"
  ],
  "nivel_de_criticidad": "alta",
  "resumen_tecnico": "La aplicación presenta problemas de conexión con PostgreSQL debido al agotamiento del pool de conexiones, provocando interrupciones del servicio."
}
```

## 9. Validación y resiliencia

El esquema Pydantic valida:

- Que la lista de tecnologías no esté vacía.
- Que el nivel de criticidad pertenezca al enum definido.
- Que el resumen técnico no esté vacío.

La cadena utiliza `with_retry` para reintentar automáticamente ante excepciones, con un máximo de tres intentos.

## 10. Testing

Ejecutar las pruebas mediante:

```bash
python test_pipeline.py
```

La prueba verifica que el pipeline devuelva un objeto `ExtraccionTecnica` con los campos requeridos.


Writing /content/drive/MyDrive/AI Engineering/ai-entity-extraction-pipeline/README.md


In [22]:
## Verificamos que están presentes todos los archivos
import os

carpeta = "/content/drive/MyDrive/AI Engineering/ai-entity-extraction-pipeline"

archivos_esperados = [
    "ai-entity-extraction-pipeline.ipynb",
    "schemas.py",
    "chain.py",
    "test_pipeline.py",
    "requirements.txt",
    "README.md"
]

for archivo in archivos_esperados:
    ruta = os.path.join(carpeta, archivo)
    estado = "OK" if os.path.isfile(ruta) else "FALTA"
    print(f"{estado}: {archivo}")

OK: ai-entity-extraction-pipeline.ipynb
OK: schemas.py
OK: chain.py
OK: test_pipeline.py
OK: requirements.txt
OK: README.md
